# Parsing GeoGebra Construction Protocolto Polars DataFrame

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
from ggblab import GeoGebra

In [3]:
# initialize base class, not open GeoGebra Widget
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [8]:
from ggblab.construction_io import ConstructionIO

In [5]:
await ggb.init()

In [6]:
c = ggb.construction.load('2025_13_01.ggb')

In [7]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [10]:
# df1 from a file df2 from a applet
df1 = await ConstructionIO.initialize_dataframe(ggb, file='2025_13_01.ggb', _columns=ConstructionIO.COLUMNS + ["ShowObject", "ShowLabel", "Auxiliary"])
df2 = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [ ]:
ConstructionIO.COLUMNS

In [11]:
df1

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""C""","""point""",null,null,null,9,true,true,false
"""A""","""point""",null,null,null,9,true,true,false
"""poly1""","""polygon""","""Polygon(C, A, 4)""",null,null,3,false,false,true
"""f""","""segment""","""Segment(C, A, poly1)""",null,null,2,false,false,true
"""g""","""segment""","""Segment(A, E, poly1)""",null,null,2,false,false,true
"""h""","""segment""","""Segment(E, D, poly1)""",null,null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""",null,null,2,false,false,true
"""E""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""",null,null,2,true,false,true


In [12]:
df2

Name,Type,Command,Value,Caption,Layer
str,str,str,str,str,i64
"""C""","""point""",null,"""C = (0, 0)""",null,9
"""A""","""point""",null,"""A = (2.2, 0)""",null,9
"""poly1""","""polygon""","""Polygon(C, A, 4)""","""poly1 = 4.6""",null,3
"""f""","""segment""","""Segment(C, A, poly1)""","""f = 2.2""",null,2
"""g""","""segment""","""Segment(A, E, poly1)""","""g = 2.2""",null,2
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2


In [13]:
set(df1["Type"].unique()) - set(df2["Type"].unique())

{'conic'}

In [14]:
set(df2["Type"].unique()) - set(df1["Type"].unique())

{'circle', 'quadrilateral', 'triangle'}

In [15]:
# df1 and df2 have different order...
# df1["Command"] == df2["Command"]
mask = df1['Command'].eq_missing(df2['Command']).not_()
# df1.filter(mask)
df2.filter(mask)

Name,Type,Command,Value,Caption,Layer
str,str,str,str,str,i64
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2
"""G""","""point""","""Polygon(A, C, 4)""","""G = (0, -2.2)""",null,4
"""H""","""point""","""Polygon(A, C, 4)""","""H = (2.2, -2.1)""",null,4
"""a_{3}""","""segment""","""Segment(G, H, poly2)""","""a_{3} = 2.2""",null,4
"""b_1""","""segment""","""Segment(H, A, poly2)""","""b_1 = 2.2""",null,4
"""proj_{u}w""","""numeric""","""(w u) / (u u)""","""proj_{u}w = 1.2""",null,8


## IR files

In [16]:
import os
os.path.splitext(ggb.file.source_file)[0]+'.json'

'2025_13_01.json'

In [19]:
df1.write_json(os.path.splitext(ggb.file.source_file)[0]+'.json')

In [17]:
import xml.etree.ElementTree as ET
root = ET.Element(c.geogebra_xml)
tree = ET.ElementTree(root)
tree.write(os.path.splitext(ggb.file.source_file)[0]+'.xml', encoding='utf-8', xml_declaration=True)